# Agent | Tools

**1. 도구 (Tool)**

https://docs.langchain.com/oss/python/integrations/tools/index#tools-and-toolkits

> "LLM이 사용할 수 있는 구체적인 기술이나 장비"

LLM은 기본적으로 학습된 데이터 내에서만 답변할 수 있으며, 실시간 정보나 정확한 수학 계산에는 취약하다. **Tool**은 이러한 한계를 보완하기 위해 LLM에게 쥐여주는 외부 기능이다.

* **역활:** 외부 API 호출, 웹 검색, 코드 실행, 파일 시스템 접근 등 LLM이 직접 할 수 없는 작업을 대신 수행한다.
* **예시:**
* `Google Search`: 최신 정보를 검색한다.
* `Calculator`: 정확한 수학 계산을 수행한다.
* `Python REPL`: 파이썬 코드를 작성하고 실행한다.



**2. 에이전트 (Agent)**

> "도구를 언제, 어떻게 사용할지 결정하는 두뇌"

**Agent**는 LLM을 추론 엔진(Reasoning Engine)으로 사용하여 사용자의 요청을 해결하기 위한 계획을 세우고 실행하는 주체이다. 단순히 정해진 코드를 순서대로 실행하는 것이 아니라, 상황에 따라 유연하게 행동을 결정한다.

* **역활:** 사용자의 질문을 분석하고, 어떤 **Tool**이 필요한지 판단(Thought)하고, 해당 도구를 실행(Action)한 뒤, 그 결과(Observation)를 보고 다음 행동을 결정하거나 최종 답변을 내놓는다.
* **작동 방식 (ReAct 패턴 예시):**
1. **질문:** "현재 서울 날씨에 맞는 옷차림 추천해줘."
2. **생각(Thought):** "서울의 현재 날씨를 먼저 알아야 한다." -> `Search` 도구 선택
3. **행동(Action):** `Search("서울 현재 날씨")` 실행
4. **관찰(Observation):** "서울 기온 5도, 맑음"이라는 결과 획득
5. **생각(Thought):** "5도면 코트나 패딩이 필요하다."
6. **최종 답변:** "현재 서울은 5도이므로 코트나 가벼운 패딩을 추천합니다."

**Agent와 Tools의 상호작용**

1. **Agent가 입력을 받음**: 사용자의 요청을 LLM으로 분석.
2. **적합한 Tool 선택**: 요청을 처리하는 데 가장 적합한 Tool을 선택.
3. **Tool 실행 및 결과 반환**: Tool을 실행하고 결과를 받아 사용자에게 응답.

In [3]:
%pip install -Uqqq langchain_openai langchain_community langchain_tavily langgraph wikipedia numexpr arxiv ddgs

Note: you may need to restart the kernel to use updated packages.


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

## Tools

In [5]:
import importlib, pkgutil # 모듈 동적 로드 / 패키지 탐색 유틸

# langchain_community.tools 패키지 로드
package = importlib.import_module('langchain_community.tools')

# 해당 패키지 경로 아래의 하위 모듈들을 하나씩 순회
for module in pkgutil.iter_modules(package.__path__):
    print(module.name) # 각 모듈(도구) 이름 출력

ainetwork
amadeus
arxiv
asknews
audio
azure_ai_services
azure_cognitive_services
bearly
bing_search
brave_search
cassandra_database
clickup
cogniswitch
connery
convert_to_openai
databricks
dataforseo_api_search
dataherald
ddg_search
e2b_data_analysis
edenai
eleven_labs
few_shot
file_management
financial_datasets
github
gitlab
gmail
golden_query
google_books
google_cloud
google_finance
google_jobs
google_lens
google_scholar
google_serper
google_trends
graphql
human
ifttt
interaction
jina_search
jira
json
memorize
merriam_webster
metaphor_search
mojeek_search
multion
nasa
nuclia
office365
openai_dalle_image_generation
openapi
openweathermap
passio_nutrition_ai
playwright
plugin
polygon
powerbi
pubmed
render
requests
riza
scenexplain
searchapi
searx_search
semanticscholar
shell
slack
sleep
spark_sql
sql_database
stackexchange
steam
steamship_image_generation
tavily_search
vectorstore
wikidata
wikipedia
wolfram_alpha
yahoo_finance_news
you
youtube
zapier
zenguard


### Wikipedia Tool


In [6]:
from langchain_community.tools import WikipediaQueryRun          # 위키피디아 질문 실행 Tool
from langchain_community.utilities import WikipediaAPIWrapper    # 위키피디아 검색/요약 API 요청 래퍼 클래스

# 위키피디아 API 래퍼를 Tool에 연결
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
print(wiki_tool.run('Physical AI')) # 위키피디아 검색/요약 결과 출력

Page: Physical artificial intelligence
Summary: Physical artificial intelligence or physical AI refers to artificial intelligence (AI) systems that perceive, reason about and act within the physical world. These systems generally combine AI models with sensors, control systems, actuators and physical machines such as robots or autonomous vehicles. Physical AI overlaps with embodied artificial intelligence, robotics and autonomous systems, but it emphasizes the complete process of perceiving an environment, motion planning an action and physically executing the task to perform work. This differs from digital AI or generative AI (GenAI), which primarily stays in the information or digital realm.
The term became increasingly prominent during the AI boom in the 2020s as AI development expanded from primarily digital applications toward humanoid robots, self-driving vehicles, smart factories and other autonomous machines. Its boundaries are not standardized, and it is often treated as a con

In [6]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pprint import pprint

messages = [('human', '걸그룹 튜이드 멤버 알려줘')]

llm = init_chat_model('gpt-5.4-mini')
# print(llm.invoke('걸그룹 튜이드 멤버 알려줘')) # 최신정보 알지 못함

agent = create_agent(
    model = llm,
    tools = [wiki_tool]
)

response = agent.invoke({'messages': messages})

pprint(response)

NameError: name 'wiki_tool' is not defined

In [8]:
print(response['messages'][-1].content)

혹시 **걸그룹 “투이드”**를 말씀하신 걸까요?  
제가 아는 범위에서는 **“튜이드/투이드”라는 이름의 걸그룹 정보가 정확히 확인되지 않아요.**

원하시면 제가 바로 찾아볼 수 있게:
- **정확한 그룹명**
- **영문 표기**
- **멤버 사진/소속사**
중 하나만 알려주세요.

원하시면 제가 **“투아이즈(Two X)”, “트와이스(TWICE)”** 같은 비슷한 이름의 그룹인지도 같이 찾아드릴게요.


### load_tools

**load_tools 사용가능 목록**

라이브러리를 통해 제공되는 외부 tool들을 langchain-community에서 통합하여 사용할 수 있다.
웬만한 기능들은 langchain 생태계 내에서 쉽게 쓸 수 있다.

https://docs.langchain.com/oss/python/integrations/providers/overview

https://docs.langchain.com/oss/python/integrations/tools

| 도구 이름        | 기능 예시               |
|------------------|------------------------|
| llm-math         | LLM 기반 수학 계산     |
| wikipedia        | 위키백과 검색          |
| serpapi          | 구글 검색 API          |
| requests_get     | HTTP GET 요청          |
| requests_post    | HTTP POST 요청         |
| arxiv            | arXiv 논문 검색        |
| pubmed           | PubMed 논문 검색       |
| dalle            | DALL-E 이미지 생성     |
| bing_search      | Bing 검색              |
| duckduckgo_search| DuckDuckGo 검색        |

### arxiv

In [9]:
import requests  # HTTP 요청 보내는 라이브러리
import xml.etree.ElementTree as ET   # XML 응답 파싱
from langchain_core.tools import tool # Langchain Tool 생성 데코레이터

@tool
def search_arxiv(arxiv_id: str) -> str:
    """ arXib 논문 ID로 제목, 저자, 초록을 조회합니다."""

    url = "https://export.arxiv.org/api/query"
    response = response.get(
        url,
        params = {
            "id_list": arxiv_id, # 논문 ID
            "max_results": 1     # 결과 1개
        },
        timeout = 10             # 응답 대기시간
    )
  
    response.raise_for_status()  # 요청 실패시 예외 발생

    root = ET.fromstring(response.text) # XML 문자열을 받아 Element 객체로 반환

    ns = {"atom": "http://www.w3.org/2005/Atom"}  #  arXiv 응답의 XML 네임 스페이스
    entry = root.find("atom:entry", ns)           # 논문 정보가 담긴 entry 태그

    if entry is None:
        return "논문 정보를 찾을 수 없습니다."

    title = entry.findtext("atom:title", namespaces = ns).strip()     # 논문 제목 추출
    summary = entry.findtext("atom:summary", namespaces = ns).strip()  # 요약 정보 추출
    authors = [  # 저자 추출
        author.findtext("atom:name", namespaces=ns)
        for author in entry.findall("atom:author", ns)
    ]

    return f"""
    제목: {title}
    저자: {', '.join(authors)}
    초록: {summary}
    """


In [10]:
from pprint import pprint

In [11]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-4.1-mini')


tools = load_tools(['wikipedia', 'llm-math'], llm = llm)

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = """
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해 주세요.
단, 숫자게산은 반드시 llm-math 도구를 사용해서 답변에 활용해야 합니다."""
)

response = agent.invoke({
    'messages': [
        ('human', '유클리드 기하에서 평행선 공준이 지켜지지 않는 기하는?')
    ]
})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1])


{'messages': [HumanMessage(content='유클리드 기하에서 평행선 공준이 지켜지지 않는 기하는?', additional_kwargs={}, response_metadata={}, id='18406a2e-8de0-49c1-a706-d4d38e35c305'),
              AIMessage(content='유클리드 기하에서 평행선 공준이 지켜지지 않는 기하는 비유클리드 기하학입니다.\n\n비유클리드 기하학은 크게 두 가지로 나뉘는데,\n1. 쌍곡기하학: 평행선 공준 대신 한 점에서 평행선을 무수히 많이 그릴 수 있다고 가정하는 기하학\n2. 타원기하학: 평행선 공준을 부정하여, 어떤 직선에 대해서도 그 직선에 평행한 직선을 그릴 수 없다고 가정하는 기하학\n\n즉, 평행선 공준이 지켜지지 않는 기하학은 비유클리드 기하학의 쌍곡기하학과 타원기하학입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 174, 'prompt_tokens': 185, 'total_tokens': 359, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_e67debc4e8', 'id': '

### duckduckgo
https://reference.langchain.com/python/langchain-community/tools/ddg_search/tool/DuckDuckGoSearchRun

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

In [12]:
# 덕덕고 검색 Tool 2종류
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

ddgs = DuckDuckGoSearchRun() # 검색 결과를 텍스트 요약 형태로 반환
print(ddgs.invoke("Trump's first name?")) # 문자열 출력

ddgs2 = DuckDuckGoSearchRun() # 검색 결과를 구조화된 리스트로 반환
print(ddgs.invoke("Trump's first name?")) # 제목/링크/스니펫 정보등의 리스트로 반환

The company name "E. Trump & Son" appeared in advertising by 1924, [44] by which year Trump ostensibly used an $800 loan from his mother to complete and sell his first house. [45][37][46] Public records, however, do not support him building until 1927, [47] the year the company was incorporated [48] (and following Trump's 21st birthday). The Trump family is a prominent wealthy American family. The best-known member is patriarch Donald Trump, the 45th and current 47th president of the United States (2017-2021, 2025-present). The Trumps are of German descent. [1] They are active in business, entertainment, politics, and real estate. Other prominent members include Donald Trump's father Fred Trump, and grandfather Frederick ... Trump was sworn in as president on January 20, 2017. During his first term, his administration focused on immigration, trade, tax cuts, and reducing government regulations. Trump withdrew the United States from the Trans-Pacific Partnership and announced that the c

In [13]:
from pprint import pprint

llm = init_chat_model(
    'gpt-5.6-luna',
    reasoning_effort='none'
)

# DDGS 도구 로드
tools = [ddgs2]

agent = create_agent(
    llm,
    tools,
    system_prompt='모르는 정보가 있으면 ddgs tool을 사용해 검색해.'
)

response = agent.invoke({
    'messages': [
        ('human', 'GS25 민음사 빵')
    ]
})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='GS25 민음사 빵', additional_kwargs={}, response_metadata={}, id='eaff1368-ddd8-4108-8dff-d08b7fcb9a5d'),
              AIMessage(content='“GS25 민음사 빵”은 **GS25와 민음사가 협업해 출시한 독서·문학 콘셉트의 베이커리 상품**을 말하는 것으로 보입니다. 다만 현재 판매 여부와 정확한 제품명·가격은 점포별로 다를 수 있어요.\n\n확인 방법:\n- **우리동네GS 앱** → 상품 검색에서 `민음사`, `민음`, `빵` 검색\n- GS25 매장 냉장·베이커리 코너 확인\n- 재고는 점포별로 달라 방문 전 앱에서 확인\n\n원하시는 게 **제품 종류·가격·판매처·후기** 중 어떤 정보인지 말씀해 주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 159, 'prompt_tokens': 179, 'total_tokens': 338, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHezyYkImmYQnA2R19ccck

tavily-search
https://docs.langchain.com/oss/python/integrations/tools/tavily_search

In [14]:
from langchain_tavily import TavilySearch # Tavily 검색 Tool

tavily_tool = TavilySearch(
    max_results = 3,
    topic = 'general',    # general/news/finance 등 선택
    include_images = True,   # 이미지 URL 함께 반환
    search_depth = 'advanced' # basic/advanced (advanced는 더 깊게 찾음)
)
tavily_tool.invoke('현재 대한민국에서 가장 핫한 이슈가 뭐야?')

{'query': '현재 대한민국에서 가장 핫한 이슈가 뭐야?',
 'follow_up_questions': None,
 'answer': None,
 'images': ['https://i.ytimg.com/vi/FfFY8UKEW5s/maxresdefault.jpg',
  'https://lookaside.instagram.com/seo/google_widget/crawler/?media_id=3957302540320513287',
  'https://lookaside.instagram.com/seo/google_widget/crawler/?media_id=3953225649355672815',
  'https://lookaside.instagram.com/seo/google_widget/crawler/?media_id=3846744411229896534',
  'https://lookaside.fbsbx.com/lookaside/crawler/threads/DX5kIoZmDNk/2/image.jpg'],
 'results': [{'url': 'https://www.instagram.com/p/DazXbWJy8SY',
   'title': 'Instagram',
   'content': "gang\\_naenge\\_official\n\n📌 대한민국에서 가장 핫한 인물 TOP10 🔥 (재미로 보는 화제성 콘텐츠)  \n  \n국내외에서 화제를 모으고 있는 인물들을 바탕으로 재미있게 구성한 TOP10입니다. 순위는 공식 조사 결과가 아닌 콘텐츠용이며, 다양한 의견이 있을 수 있습니다.  \n  \n🏆 TOP10  \n🥇 BTS  \n🥈 블랙핑크  \n🥉 홍명보  \n4️⃣ 손흥민  \n5️⃣ 이강인  \n6️⃣ 리센느(RESCENE)  \n7️⃣ 에이티즈(ATEEZ)  \n8️⃣ 조국  \n9️⃣ 이재명  \n🔟 윤석열  \n  \n💬 여러분이 생각하는 현재 대한민국에서 가장 핫한 인물은 누구인가요?  \n댓글로 자유롭게 의견을 남겨주세요!  \n  \n#해시

In [15]:
llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# DDGS 도구 로드
tools = [tavily_tool]

agent = create_agent(
    llm,
    tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답하시오.'
)

response = agent.invoke({
    'messages': '현재 AI업계에서 가장 핫한 주제는?'})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='현재 AI업계에서 가장 핫한 주제는?', additional_kwargs={}, response_metadata={}, id='d4248fa2-e337-47db-99f7-df9ecb63375d'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 1310, 'total_tokens': 1372, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHf08t3XvpPTNdHK1nM4RaGwRpOpj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045d3-921e-7563-8aaf-862714e51a59-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current hottest topics in AI industry 2026 tr

In [16]:
llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# DDGS 도구 로드
tools = [tavily_tool]

agent = create_agent(
    llm,
    tools,
    system_prompt='''당신은 미국주식시장 분석봇입니다.
사용자가 요청한 기업에 대한 2026년 보고서를 직관적으로 분석해주세요.

# 출력형식
다음 내용을 포함해 표형식 출력 (분석기관별 레코드로 작성)

1. 분석기관명
2. 목표주가범위 (최저 ~ 최대)
3. 전망근거 키워드
4. 신뢰도 지수(1 ~ 10)당신은 현명한 챗봇입니다''')


response = agent.invoke({
    'messages': '2026년 상승할 가능성이 가장 높은 미국주식은?'})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='2026년 상승할 가능성이 가장 높은 미국주식은?', additional_kwargs={}, response_metadata={}, id='b056a2b6-7c48-47df-8994-e2483c4c9828'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 1394, 'total_tokens': 1449, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHf0JX1iLBlbKxCHXGRSIt5e9ntdB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045d3-c983-79d2-b77d-a8ffad760c0c-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '2026 year ahead Wall Street target pri

In [17]:
from IPython.display import display, Markdown

display(Markdown(response['messages'][-1].content))

2026년 **상승 가능성이 가장 높은 미국주식**을 “애널리스트 목표주가 상향폭 + 실적 모멘텀 + AI/클라우드 수혜” 기준으로 보면, **마이크론(MU)**가 가장 강하게 보입니다.  
다만 “가장 높은 상승 가능성”은 기준에 따라 달라질 수 있어, 아래는 **주요 기관별 2026년 관점 분석표**로 정리했습니다.

| 분석기관명 | 목표주가범위 (최저 ~ 최대) | 전망근거 키워드 | 신뢰도 지수(1~10) |
|---|---:|---|---:|
| Stifel | $300 ~ $550 | HBM 수요, AI 메모리 슈퍼사이클, DRAM 가격 상승, 공급부족 | 9 |
| RBC Capital | $320 ~ $330 | AI 인프라 투자, 클라우드 수요, 메모리 업사이클, 실적 가시성 | 8 |
| HSBC | $350 ~ $500 | 메모리 가격 회복, AI 서버 수요, 실적 추정치 상향 | 8 |
| KeyBanc / KeyCorp | $325 ~ $450 | 고대역폭 메모리(HBM), 데이터센터 수요, 마진 개선 | 8 |
| Amazon 관련 일부 애널리스트 | $280 ~ $340 | AWS 성장, AI 인프라 ROIC, 광고/클라우드 동시 성장 | 7 |
| Microsoft 관련 일부 애널리스트 | $625 ~ $650 | Azure AI, Copilot 확산, 엔터프라이즈 AI 채택, 고마진 서비스 | 7 |

### 한줄 결론
- **공격적 상승 여력 1순위:** **Micron (MU)**
- **안정성과 대형주 상승 모멘텀:** **Microsoft (MSFT)**
- **클라우드/AI 복합 성장:** **Amazon (AMZN)**

### 제 의견
- **2026년 “가장 크게 오를 확률”**만 보면: **MU**
- **리스크 대비 안정적 상승**을 원하면: **MSFT**
- **장기 성장성과 생태계 확장**을 원하면: **AMZN**

원하시면 다음 단계로  
**“2026년 유망 미국주식 TOP 10”**을 상승확률 기준으로 표로 정리해드릴게요.

In [18]:
# eval /exec로 문자열 코드 실행
a = 10
print(eval("5+ 3 + a"))
exec("b = 10")
print(b)

18
10


In [19]:
from langchain_core.tools import tool

@tool
def simple_calculator(query: str) -> str:
    """
    산술연산을 위한 간단한 계산기 Tool
    Args:
        query: 계산식
    return
        계산식 결과값
        
    Examples:
    - simple_calculator("5 + 3 - 2") -> "계산 결과: 6"
    -simple_calculator("4 ** 2 / 8) -> 계산 결과: 2
    """

    try:
        result = eval(query)            # 문자열을 eval로 평가(결과 반환)
        return f"계산 결과 : {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"

simple_calculator


StructuredTool(name='simple_calculator', description='산술연산을 위한 간단한 계산기 Tool\nArgs:\n    query: 계산식\nreturn\n    계산식 결과값\n\nExamples:\n- simple_calculator("5 + 3 - 2") -> "계산 결과: 6"\n-simple_calculator("4 ** 2 / 8) -> 계산 결과: 2', args_schema=<class 'langchain_core.utils.pydantic.simple_calculator'>, func=<function simple_calculator at 0x0000024543F068E0>)

In [20]:
llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# DDGS 도구 로드
tools = [tavily_tool]

agent = create_agent(
    llm,
    tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답하시오.'
)

response = agent.invoke({
    'messages': '7+3 *8?'})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='7+3 *8?', additional_kwargs={}, response_metadata={}, id='727ffa6b-7472-4e01-a576-63a99a052c64'),
              AIMessage(content='7 + 3 × 8 = 31\n\n곱셈을 먼저 해서, **3 × 8 = 24**, 그다음 **7 + 24 = 31** 입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 1303, 'total_tokens': 1348, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHf0deQcOuTmZVUfUvlcWvAbX5cyO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a045d4-17b1-7603-9ef7-73198dd5875c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata

In [21]:
import os
import json
import requests

from langchain_core.tools import tool


OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')


@tool
def get_current_weather(city_name='seoul', units='metric'):
    """
    OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        - city_name: str 날씨 정보를 가져올 도시 이름. 반드시 영문으로 작성하세요.
            - 변환 예시:
                - 서울 -> Seoul
                - 충남, 충청남도 -> Chungcheongnam-do
                - 부산 -> Busan
        - units: str 온도 단위를 설정하는 문자열
            - metric(기본값: 섭씨, 미터)
            - imperial(화씨, 야드)

    Return:
        - str: JSON 형식으로 변환된 현재 날씨 정보
    """

    url = (
        f'https://api.openweathermap.org/data/2.5/weather'
        f'?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}'
    )

    response = requests.get(url)
    data = response.json()
    weather_info = {}

    if response.status_code == 200:
        weather_description = data['weather'][0]['description']
        temp = data['main']['temp']
        temp_feels_like = data['main']['feels_like']
        humidity = data['main']['humidity']

        weather_info = {
            'city': city_name,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }

    else:
        weather_info = {
            'city': city_name,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like': 'Not Found',
            'humidity': 'Not Found'
        }

    return json.dumps(weather_info, ensure_ascii=False)


get_current_weather



StructuredTool(name='get_current_weather', description='OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수\n\nArgs:\n    - city_name: str 날씨 정보를 가져올 도시 이름. 반드시 영문으로 작성하세요.\n        - 변환 예시:\n            - 서울 -> Seoul\n            - 충남, 충청남도 -> Chungcheongnam-do\n            - 부산 -> Busan\n    - units: str 온도 단위를 설정하는 문자열\n        - metric(기본값: 섭씨, 미터)\n        - imperial(화씨, 야드)\n\nReturn:\n    - str: JSON 형식으로 변환된 현재 날씨 정보', args_schema=<class 'langchain_core.utils.pydantic.get_current_weather'>, func=<function get_current_weather at 0x0000024543ED5940>)

In [22]:
llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# 도구 로드
tools = [simple_calculator, get_current_weather]

agent = create_agent(
    llm,
    tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답하시오.'
)

response = agent.invoke({
    'messages': [
        ('human', '강원도 사는데 오늘 옷 뭐 입을까?')
    ]
})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='강원도 사는데 오늘 옷 뭐 입을까?', additional_kwargs={}, response_metadata={}, id='d353db8b-0542-4570-b6e3-32197de6a133'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 401, 'total_tokens': 426, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHf0fhB53qMmCHYZSPcMj3ZPCdtKn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045d4-1cdc-7831-a005-ce226e6856c2-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city_name': 'Gangwon-do', 'units': 'metric'}, 'id':

In [23]:
!pip install pytz


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
# 한국 기준 현재 날짜/시간을 반환하는 Tool
from datetime import datetime
from pytz import timezone
from langchain_core.tools import tool


@tool
def get_current_datetime(
    format: str = '%Y-%m-%d %H:%M:%S'
) -> str:
    """
    한국 기준 현재 시각 정보를 반환하는 함수

    Args:
        format: 날짜/시각 형식 지정

    Return:
        현재 시각 문자열

    get_current_datetime() -> "2026-01-15 12:18:32"
    """

    kst = timezone('Asia/Seoul') # 한국 시간대(KST) 설정

    return datetime.now(kst).strftime(format) # 현재 서울 시간을 받아, format 형식의 문자열로 변환


get_current_datetime

StructuredTool(name='get_current_datetime', description='한국 기준 현재 시각 정보를 반환하는 함수\n\nArgs:\n    format: 날짜/시각 형식 지정\n\nReturn:\n    현재 시각 문자열\n\nget_current_datetime() -> "2026-01-15 12:18:32"', args_schema=<class 'langchain_core.utils.pydantic.get_current_datetime'>, func=<function get_current_datetime at 0x0000024543E3AE80>)

In [25]:
@tool
def calculate_age(today_date: str, bitrh_date: str) -> int:
    """
    오늘날짜, 생년월일을 입력받아 나이를 계산하는 도구
    Args:
        - today_date(str): 오늘 날짜 (yyyy-mm-dd형식)
        - birth_date(str): 생년월일 (yyyy-mm-dd형식)
    Return:
        - 계산된 만나이(int)
    """

    try:
        today = datetime.strptime(today_date, '%Y-%m-%d')    # 오늘 날짜 문자열 -> datetime 변환
        birthday = datetime.strptime(bitrh_date, '%Y-%m-%d') # 생일 날짜 문자열 -> datetime 변환

        age = today.year - birthday.year # 기본 나이 계산
        # 생일이 아직 안지난 경우
        if (today.month, today.day) < (birthday.month, birthday.day):
            age -= 1 # 만나이는 -1
        return age
    except ValueError:
        return "날짜 형식이 올바르지 않습니다. yyyy-mm-dd 형식으로 전달해 주세요."

calculate_age.invoke({'today_date': '2026-08-27', 'bitrh_date': '1920-10-11'})

105

In [26]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools
from pprint import pprint

In [27]:
from langchain_community.agent_toolkits import load_tools

llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# Wikipedia 및 사용자 정의 도구 로드
tools = load_tools(['wikipedia']) + [
    get_current_datetime,
    calculate_age
]

agent = create_agent(
    llm,
    tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답하시오.'
)

response = agent.invoke(
    {
        'messages': [
            ('human', '트럼프 대통령은 몇 살이야?')
        ]
    },
    config={
        'recursion_limit': 10
    }
)

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content) # 30명이 api 요청해서 응답이 안옴 ㅠㅠ

TypeError: 'module' object is not callable

## Memory
agent의 checkpointer속성에 메모리객체를 대화내역을 저장한다.
- 임시저장 InMemorySaver()
- 영구저장 SqliteSaver()

### InMemorySaver

In [ ]:
from langchain_tavily import TavilySearch

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  # 대화 상태 저장

llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

tools = [TavilySearch()]

agent = create_agent(
    llm,
    tools,
    checkpointer=InMemorySaver()
)

response = agent.invoke(
    input={
        'messages': [
            ('human', '안녕! 만나서 반갑다! 나는 cap이라고 해. 넌 누구니?')
        ]
    },
    config={
        'configurable': {
            'thread_id': '100'
        }
    }
)
print()

In [ ]:
response = agent.invoke(
    input = {'messages': [('human', '나는 패왕 항우다')]},
    config = {'configurable': {'thread_id': '200'}} # thread_id 200번으로 새로운 대화 시작
)
print(response['messages'][-1].content)

### sqliteSaver

In [ ]:
# Langgraph 상태 저장을 SQLite로 영속화하여 저장하는 체크포인터 패키지
%pip install -Uqqq langgraph-checkpoint-sqlite

In [ ]:
# 사용자가 재접속한 상황
from langgraph.checkpoint.sqlite import SqliteSaver # Sqlite 기반 체크포인트(Saver)
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

# Sqlite DB 연결을 checkpoint.db 컨텍스트로 관리
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup() # 테이블 생성 및 초기화

    # 에이전트가 상태 저장소로 checkpointer 활용
    agent = create_agent(llm,tools,checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human','오케이. 완전 이해했어! 그럼 니가 말해준 langchain, langgraph를 세줄요약해줘')]},
        config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
    )

    pprint(response)
    print("=" * 50)
    pprint(response['messages'][-1].content)

In [ ]:
# SQLite 체크포인터(DB)의 특정 thread_id의 대화 메시지 조회
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer_tuple = checkpointer.get_tuple({"configurable": {"thread_id": "100"}})

    checkpointer_data = checkpointer_tuple.checkpoint
    messages = checkpointer_data['channel_values']['messages']

    for i, message in enumerate(messages, 1):
        msg_type = getattr(message, 'type', message.__class__.__name__)
        print(f"{i}: [{msg_type}] {message.content}")
        print()

## Middleware
https://docs.langchain.com/oss/python/langchain/middleware

미들웨어를 통해 에이전트의 추론 과정 중간에 개입하여 내부 동작을 커스터마이징할 수 있다.
Agent를 세부적으로 커스터마이징하기 위한 대부분의 작업을 미들웨어로 할 수 있다.

- 대화 기록 요약
- 동작 중 사용자 입력 대기
- 특정 모델 또는 tool에 대한 호출 제약
- fallback
- PII(개인식별정보) 처리 등

### SummarizationMiddleware

In [29]:
from langgraph.checkpoint.memory import InMemorySaver

In [31]:
from langchain.agents.middleware import SummarizationMiddleware # 대화내역 자동 요약 미들웨어

model = init_chat_model('gpt-5.4-mini') # 메인 에이전트 LLM (상대적으로 성능이 좋고 비싼모델)
summary_model = init_chat_model('gpt-5,4-mini') # 요약 모델 (상대적으로 싼모델)
middleware_summarize = SummarizationMiddleware(
    model = summary_model,         # 요약모델
    trigger = ('tokens',1000),     # 누적 노큰 1000 이상일 때 트리거
    keep = ('messages', 1),        # 최근 1개 메시지는 요약 제외
    summary_prompt = '다음 대화 내용을 적절하게 요약해주세요.\n{messages}'
)

agent = create_agent(
    model = model,
    tools = [],
    checkpointer = InMemorySaver(),    # 메모리 저장소
    middleware = [middleware_summarize] # 요약
)

In [32]:
response = agent.invoke(
    input = {'messages':[('human','뮤지컬 wicked의 내용을 Elphaba 입장에서 서술해줘. Elphaba역으로 연극에 출연해야되서 준비중이야. 중요한 포인트들을 다 짚어줘!')]},
    config = {'configurable': {'thread_id': '5'}}
)

pprint(response)
print("="*50)
pprint(response['messages'][-1].content)
print("="*100)

response = agent.invoke(
    input = {'messages':[('human','Elphaba의 심정을 어떻게 표현할까?')]},
    config = {'configurable': {'thread_id': '5'}}
)

pprint(response)
print("="*50)
pprint(response['messages'][-1].content)
print("="*100)

response = agent.invoke(
    input = {'messages':[('human','메소드 연기가 필요한 부분이 있을까?')]},
    config = {'configurable': {'thread_id': '5'}}
)

pprint(response)
print("="*50)
pprint(response['messages'][-1].content)
print("="*100)

{'messages': [HumanMessage(content='뮤지컬 wicked의 내용을 Elphaba 입장에서 서술해줘. Elphaba역으로 연극에 출연해야되서 준비중이야. 중요한 포인트들을 다 짚어줘!', additional_kwargs={}, response_metadata={}, id='549b826d-e395-4471-b40b-f465e3fea218'),
              AIMessage(content='물론이야. **Elphaba 입장에서 본 *Wicked*의 줄거리와 감정선**을, 네가 무대에서 바로 사용할 수 있게 **장면별 핵심 포인트 + 감정/목표 + 연기 포인트** 중심으로 정리해줄게.  \n(※ 스포일러 포함)\n\n---\n\n# 1) Elphaba의 전체 이야기 한 줄 요약\n**“나는 처음엔 사랑받고 싶었고, 인정받고 싶었고, 세상을 바꾸고 싶었지만, 결국 세상은 나를 괴물로 만들었고, 나는 내 방식대로 진실과 자유를 택한다.”**\n\nElphaba는 단순히 “착한 마녀/나쁜 마녀”가 아니라,  \n**상처받고, 분노하고, 사랑하고, 희생하고, 결국 오해받는 혁명가**야.\n\n---\n\n# 2) Elphaba의 감정 아크\nElphaba는 처음부터 끝까지 같은 성격이 아니고, 크게 이런 흐름으로 변해:\n\n1. **외로움 / 경계심**  \n2. **희망 / 가능성**  \n3. **사랑 / 설렘**  \n4. **배신 / 분노**  \n5. **각성 / 저항**  \n6. **체념처럼 보이는 결단**  \n7. **자기 선택으로서의 해방**\n\n즉, 그녀의 핵심은  \n**“인정받고 싶다” → “사랑하고 싶다” → “정의롭게 살고 싶다” → “그러나 세상이 그걸 허락하지 않는다”**야.\n\n---\n\n# 3) 장면별 Elphaba 입장 정리\n\n## A. 오프닝: 등장 자체가 이미 ‘타자’로서의 삶\nElphaba는 태어날 때부터 초록 피부 때문에 타인과 다르게 취급받아.  \n어린 시절부터 가족에게도 완

OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}}

### PIIMiddleware
**PII (Personally Identifiable Information) 개인식별정보 처리**

https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection

In [4]:
from langchain.agents import create_agent

In [7]:
from langchain.agents.middleware import PIIMiddleware # PII(개인정보) 탐지/처리 미들웨어

middleware_email = PIIMiddleware(
    pii_type = 'email', # 이메일 PII 처리
    detector = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+.[a-zA-Z]{2,}", # +: 1글자 이상, {2,}: 2글자 이상
    strategy = 'redact',   # 삭제처리
    apply_to_input = True
)

middleware_credit_card = PIIMiddleware(
    pii_type = 'credit_card', # 신용카드 PII 처리
    detector = r'(?:\d{4}[-\s]?){3}\d{4}|\d{4}[-\s]?\d{6}[-\s]?\d{5}', # 카드번호 16자리 표현식
    strategy = 'mask',    # 마스킹 처리
    apply_to_input = True
)

middleware_api_key = PIIMiddleware(
    pii_type = 'api_key',   # API 키 PII 처리
    detector = r"sk-[a-zA-Z0-9-_]{161}", # OpenAI 키 예시
    strategy = 'mask'
)

agent = create_agent(
    model = init_chat_model('gpt-5.4-mini'),
    tools = [],
    middleware = [middleware_email, middleware_credit_card, middleware_api_key]
)

In [11]:
response = agent.invoke({
    'messages': [
        (
            'human',
            'AWS를 사용하는데 요금이 잘못 청구된 것 같아 ㅠㅠ '
            '금액 재조정 요청 메일 주소는 capybara@gmail.com이야.'
        )
    ]
})

pprint(response)

{'messages': [HumanMessage(content='AWS를 사용하는데 요금이 잘못 청구된 것 같아 ㅠㅠ 금액 재조정 요청 메일 주소는 [REDACTED_EMAIL]이야.', additional_kwargs={}, response_metadata={}, id='638e31ef-6c1f-4d3c-83fa-f5801d355f42'),
              AIMessage(content='물론이야. 아래처럼 AWS 청구 금액 재조정(조정/검토) 요청 메일을 보내면 돼.  \n받는 사람: **[REDACTED_EMAIL]**\n\n---\n\n**제목:** AWS 청구 금액 재조정 요청드립니다\n\n안녕하세요, AWS 지원팀 담당자님.\n\nAWS 사용 중 청구된 금액에 대해 확인해보니, 일부 항목이 잘못 청구된 것으로 보여 재조정을 요청드리고자 메일드립니다.\n\n아래 내용 확인 부탁드립니다.\n\n- **계정 ID:** [AWS Account ID]\n- **청구 기간:** [YYYY-MM-DD ~ YYYY-MM-DD]\n- **문제 내역:** [잘못 청구된 것으로 보이는 서비스/요금 항목]\n- **요청 사항:** 해당 청구 금액 검토 후 재조정 부탁드립니다.\n\n필요하신 경우 추가 자료나 상세 내역도 전달드리겠습니다.  \n확인 부탁드리며, 회신 기다리겠습니다.\n\n감사합니다.  \n[이름]  \n[회사명/조직명]  \n[연락처]\n\n---\n\n원하면 내가 이걸 **더 정중한 버전**이나 **영문 버전**으로도 바로 써줄게.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 271, 'prompt_tokens': 44, 'total_tokens': 315, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasonin

In [13]:
response = agent.invoke({
    'messages':[('human','환경변수 설정을 잘못한 것 같다. ')]
})

pprint(response)

{'messages': [HumanMessage(content='환경변수 설정을 잘못한 것 같다. ', additional_kwargs={}, response_metadata={}, id='15718db6-a0b9-406b-9942-2f5c0fe58920'),
              AIMessage(content='어떤 환경변수인지에 따라 원인이 많이 달라져요.  \n지금 상태를 보면 **환경변수 설정이 잘못됐거나, 적용이 안 된 경우**로 보입니다.\n\n빠르게 확인할 수 있는 것들:\n\n1. **값이 실제로 들어갔는지 확인**\n   - Linux/macOS:\n     ```bash\n     echo $변수명\n     ```\n   - Windows PowerShell:\n     ```powershell\n     echo $env:변수명\n     ```\n\n2. **설정 파일 문법 확인**\n   - `export VAR=value` 형태인지\n   - `VAR=value`만 적고 `export`를 안 했는지\n   - 따옴표, 공백, 특수문자 때문에 깨지지 않았는지\n\n3. **적용 범위 확인**\n   - 현재 터미널에서만 설정한 건지\n   - `.bashrc`, `.zshrc`, `.env`, 시스템 환경변수 중 어디에 넣었는지\n   - 수정 후 새 터미널을 열거나 `source`를 했는지\n\n4. **오타 확인**\n   - 변수명 철자\n   - 대소문자\n   - 앞뒤 공백\n\n5. **실행 중인 프로세스에 반영됐는지**\n   - 이미 켜져 있던 앱은 새 환경변수를 못 볼 수 있어요.  \n     이 경우 재시작이 필요합니다.\n\n원하시면 제가 같이 봐드릴게요.  \n아래 중 하나를 보내주시면 원인 좁혀드릴 수 있습니다:\n\n- 사용한 OS: Linux / macOS / Windows\n- 설정한 환경변수 이름과 값\n- 설정한 방법: `.env`, `.bashrc`, `.zshrc`, Docker, system

### Streaming
openai모델은 조직인증된 사용자에 한해서 stream기능을 사용할수 있다.

In [17]:
model = init_chat_model('gpt-5.4-mini')
agent = create_agent(model)
stream = agent.stream(
    input = {'messages': [('human', '마이클잭슨은 영향력 기준 역사상 세계 1위 아티스트인가?')]},
    stream_mode = 'messages'
)

for chunk, metadata in stream:
    print(chunk.content, end='', flush = True)

짧게 말하면 **“세계 1위”라고 단정할 수는 없지만, 역사상 가장 영향력 있는 아티스트 후보 중 최상위권**인 건 맞습니다.

핵심은 **“영향력”을 무엇으로 보느냐**예요.

### 마이클 잭슨이 최상위권인 이유
- **팝 음악의 글로벌 표준을 바꿈**
- **뮤직비디오를 예술·마케팅의 핵심 도구로 격상**
- **댄스, 퍼포먼스, 무대 연출의 기준을 바꿈**
- **흑인 아티스트의 세계적 대중성 확대에 큰 영향**
- **세계 각국의 음악, 패션, 광고, 엔터테인먼트 전반에 영향**

### 하지만 “역사상 1위”라고 단정하기 어려운 이유
- 영향력은 **시대·장르·지역·매체**에 따라 달라짐
- 예를 들어
  - **비틀즈**: 대중음악 전체의 방향을 바꾼 영향
  - **엘비스 프레슬리**: 초기 록앤롤 대중화
  - **밥 딜런**: 가사와 송라이팅의 혁신
  - **마이클 잭슨**: 퍼포먼스와 팝 문화의 세계화
- 즉, **누가 “최고”인지 객관적 합의는 없음**

### 결론
- **대중음악/팝문화 기준으로는 마이클 잭슨을 역사상 1, 2위를 다투는 인물로 보는 견해가 매우 강함**
- 하지만 **“역사상 세계 1위 아티스트”라는 절대적 결론은 주관적**입니다

원하시면 제가  
**“영향력 기준 TOP 10 아티스트”**를 기준별로 나눠서 정리해드릴게요.